In [ ]:
# =========================
# IMPORTS
# =========================
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, datasets, utils


# =========================
# KAGGLE SETUP
# =========================

# Install Kaggle API
!pip install -q kaggle

# Upload kaggle.json (API key)
from google.colab import files
files.upload()

# Configure Kaggle credentials
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip dataset
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip


# =========================
# BASIC CNN - DATA LOADING
# =========================

# Load training dataset
train_ds = utils.image_dataset_from_directory(
    "/content/seg_train/seg_train",
    image_size=(128, 128),
    batch_size=32
)

# Print class names
print(train_ds.class_names)

# Inspect one batch
for images, labels in train_ds.take(1):
    print(images.shape)
    print(labels.shape)

# Visualize one sample image
img = images[31]
label = labels[31]

print(img.shape)

plt.imshow(img.numpy().astype("uint8"))
plt.title(train_ds.class_names[label])
plt.show()


# =========================
# VALIDATION DATA
# =========================

val_ds = utils.image_dataset_from_directory(
    "/content/seg_test/seg_test",
    image_size=(128, 128),
    batch_size=32,
    shuffle=False
)

# Inspect validation batch
for images, labels in val_ds.take(1):
    print(images.shape)
    print(labels.shape)

# Visualize validation sample
img = images[30]
label = labels[30]

print(img.shape)

plt.imshow(img.numpy().astype("uint8"))
plt.title(val_ds.class_names[label])
plt.show()


# =========================
# DATA PIPELINE OPTIMIZATION
# =========================

# Prefetch for faster training
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)


# =========================
# BASIC CNN MODEL
# =========================

basic_cnn = models.Sequential([

    # Data augmentation
    layers.RandomFlip("horizontal", input_shape=(128, 128, 3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    # Normalize pixel values
    layers.Rescaling(1./255),

    # --- Block 1 ---
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # --- Block 2 ---
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # --- Block 3 ---
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    # Convert feature maps → vector
    layers.GlobalAveragePooling2D(),

    # Classification head
    layers.Dense(128),
    layers.Activation('relu'),
    layers.Dense(6, activation='softmax')
])

# Model summary
basic_cnn.summary()

# Compile model
basic_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train model
history = basic_cnn.fit(train_ds, epochs=15, validation_data=val_ds)

# Evaluate model
val_loss, val_acc = basic_cnn.evaluate(val_ds)
print("Basic CNN Validation Accuracy:", val_acc)


# =========================
# TRAINING CURVES (CNN)
# =========================

# Accuracy plot
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Basic CNN Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['train', 'validation'])
plt.show()

# Loss plot
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Basic CNN Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['train', 'validation'])
plt.show()


# =========================
# PREDICTIONS (CNN)
# =========================

# Load prediction dataset
pred_ds = utils.image_dataset_from_directory(
    "/content/seg_pred",
    image_size=(128, 128),
    batch_size=32,
    shuffle=False
)

# Predict
pred_basic_cnn = basic_cnn.predict(pred_ds)

# Convert probabilities → class indices
pred_labels = np.argmax(pred_basic_cnn, axis=1)

# Class names
class_names = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

# Convert to readable labels
pred_classes = [class_names[i] for i in pred_labels]

# Show predictions
for images, _ in pred_ds.take(1):
    for i in range(5):
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(pred_classes[i])
        plt.axis("off")
        plt.show()


# =========================
# VGG SECTION
# =========================

vgg_style = models.Sequential([

    layers.RandomFlip("horizontal", input_shape=(128, 128, 3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    layers.Rescaling(1./255),

    # --- Block 1 ---
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.MaxPooling2D((2, 2)),

    # --- Block 2 ---
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.MaxPooling2D((2, 2)),

    # --- Block 3 ---
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128),
    layers.Activation('relu'),

    layers.Dropout(0.3),

    layers.Dense(6, activation='softmax')
])

# Compile and train
vgg_style.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history1 = vgg_style.fit(train_ds, epochs=15, validation_data=val_ds)

# Evaluate
vgg_loss, vgg_acc = vgg_style.evaluate(val_ds)
print("VGG Validation Accuracy:", vgg_acc)


# =========================
# PREDICTIONS (VGG)
# =========================

pred_vgg = vgg_style.predict(pred_ds)
pred_vgg_labels = np.argmax(pred_vgg, axis=1)
pred_vgg_classes = [class_names[i] for i in pred_vgg_labels]

for images, _ in pred_ds.take(1):
    for i in range(5):
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(pred_vgg_classes[i])
        plt.axis("off")
        plt.show()


# =========================
# RESNET SECTION
# =========================

# Residual block with skip connection
def residual_block(x, filters, downsample=False):

    shortcut = x
    stride = 2 if downsample else 1

    x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Adjust shortcut if dimensions mismatch
    if downsample or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), strides=stride)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)

    return x


# Build ResNet model
def conv_model(input_shape):

    inputs = layers.Input(input_shape)

    # Data augmentation
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)

    x = layers.Rescaling(1./255)(x)

    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Residual stacks
    x = residual_block(x, 32, True)
    x = residual_block(x, 32)

    x = residual_block(x, 64, True)
    x = residual_block(x, 64)

    x = residual_block(x, 128, True)
    x = residual_block(x, 128)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(6, activation='softmax')(x)

    return models.Model(inputs, outputs)


# Train ResNet
resnet_model = conv_model((128, 128, 3))

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = resnet_model.fit(train_ds, epochs=15, validation_data=val_ds)

# Evaluate
resnet_loss, resnet_acc = resnet_model.evaluate(val_ds)
print("ResNet Accuracy:", resnet_acc)


# =========================
# PREDICTIONS (RESNET)
# =========================

pred_resnet = resnet_model.predict(pred_ds)
pred_resnet_labels = np.argmax(pred_resnet, axis=1)
pred_resnet_classes = [class_names[i] for i in pred_resnet_labels]

for images, _ in pred_ds.take(1):
    for i in range(5):
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(pred_resnet_classes[i])
        plt.axis("off")
        plt.show()